# Module 5: Biosecurity, Ethics & The Future
## Lab: Computational Biosecurity — Screening, Watermarking & The Pitch

**Lagos Bio-Design Bootcamp** | [JobAiReady](https://github.com/JobAiReady/lagos-bio-design)

---

### Objective
Use GPU-powered protein language models to audit your capstone designs: detect AI-generated sequences, screen against known threat proteins using ESM embeddings, validate final candidates with ESMFold, and compile results into a startup pitch.

### Prerequisites
- Completed Modules 1–4 (especially the Capstone)
- **GPU Runtime enabled** (Runtime → Change runtime type → T4 GPU)

### Deliverable
- Biosecurity screening report for your capstone candidates
- A 5-minute pitch deck (PDF or slides)

### Why GPU?
We use **ESM-2** (Meta's 650M-parameter protein language model) to compute sequence embeddings and detect AI-designed proteins. We also re-validate top candidates with **ESMFold** for final structural QC.

> ⚠️ **GPU Required.** Go to Runtime → Change runtime type → T4 GPU.

---
## Setup

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU not found! Enable T4 GPU in Runtime settings.'
print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip install -q fair-esm biopython matplotlib numpy pandas scipy

import os
import hashlib
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy.spatial.distance import cosine

print('All packages installed.')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')

---
## Part 1: ESM-2 Protein Language Model — Loading the 650M Parameter Model

ESM-2 is Meta AI's protein language model trained on ~250 million protein sequences. We'll use it to:
1. **Compute sequence embeddings** — a numerical fingerprint for any protein
2. **Detect AI-designed vs natural proteins** — AI designs have distinctive embedding signatures
3. **Screen sequences against known threats** — by comparing embeddings to dangerous proteins

This requires GPU due to the model size (650M parameters, ~2.5 GB VRAM).

In [ ]:
import esm

# Load ESM-2 (650M parameters — requires GPU)
print('Loading ESM-2 650M model... (this takes ~30 seconds)')
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model = model.eval().cuda()
batch_converter = alphabet.get_batch_converter()
print(f'ESM-2 loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')

def get_embedding(sequence, label='seq'):
    """Compute the mean ESM-2 embedding for a protein sequence."""
    data = [(label, sequence)]
    _, _, tokens = batch_converter(data)
    tokens = tokens.cuda()
    
    with torch.no_grad():
        results = model(tokens, repr_layers=[33], return_contacts=False)
    
    # Mean pool over sequence length (exclude BOS/EOS tokens)
    embedding = results['representations'][33][0, 1:len(sequence)+1].mean(dim=0)
    return embedding.cpu().numpy()

# Test with a short sequence
test_emb = get_embedding('MKTLLILAVFCLASQCGA')
print(f'Embedding shape: {test_emb.shape}')
print(f'Embedding norm: {np.linalg.norm(test_emb):.2f}')

In [ ]:
---
## Part 2: AI-Design Detection — The FoldMark Concept (GPU)

Can we tell if a protein was designed by AI or evolved naturally? The **FoldMark** concept proposes that AI-generated proteins carry detectable signatures in their embedding space.

**Method:** Compare ESM-2 embeddings of known natural proteins vs known AI-designed proteins. AI designs tend to cluster differently in embedding space due to the biases of generative models.

We'll build a simple detector using cosine similarity to reference sets.

# Reference sets: natural proteins vs AI-designed sequences
# Natural: well-known proteins from PDB/UniProt
natural_proteins = {
    'Insulin (Human)':        'MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKT',
    'Lysozyme (Hen)':         'KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL',
    'GFP (Jellyfish)':        'MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK',
    'Ubiquitin (Human)':      'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
    'Hemoglobin alpha':       'MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH',
    'Cytochrome C (Horse)':   'GDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKERADLIAYLKKATNE',
}

# AI-designed: sequences from generative models (RFDiffusion/ProteinMPNN-like)
# These are example hallucinated sequences with characteristics typical of AI design
ai_designed = {
    'RFDiff_binder_01':  'EEEELKKELKKELKKELKKELKKELKKELKKELKKELKKELKKELKKELKKELKKEEEE',
    'RFDiff_binder_02':  'DAAARKAAARKAAARKAAARKAAARKAAARKAAARKAAARKAAARKAAARKAAARKDDD',
    'MPNN_redesign_01':  'SIEELKKEIEELKKEIEELKKEIEELKKEIEELKKEIEELKKEIEELKKEIEELKKS',
    'MPNN_redesign_02':  'TLLEKLLEKLLEKLLEKLLEKLLEKLLEKLLEKLLEKLLEKLLEKLLEKLLEKLLET',
    'Halluc_scaffold_01': 'AEAAAKEAAAKEAAAKEAAAKEAAAKEAAAKEAAAKEAAAKEAAAKEAAAKEAAAKEA',
    'Halluc_scaffold_02': 'RLEELRREELRREELRREELRREELRREELRREELRREELRREELRREELRRELL',
}

print(f'Natural reference set: {len(natural_proteins)} proteins')
print(f'AI-designed reference set: {len(ai_designed)} proteins')
print('Computing ESM-2 embeddings for all reference sequences...')

# Compute embeddings (GPU-intensive)
natural_embeddings = {}
for name, seq in natural_proteins.items():
    natural_embeddings[name] = get_embedding(seq, name)
    print(f'  ✓ {name} ({len(seq)} aa)')

ai_embeddings = {}
for name, seq in ai_designed.items():
    ai_embeddings[name] = get_embedding(seq, name)
    print(f'  ✓ {name} ({len(seq)} aa)')

# Compute centroids
natural_centroid = np.mean(list(natural_embeddings.values()), axis=0)
ai_centroid = np.mean(list(ai_embeddings.values()), axis=0)

print(f'\nNatural centroid norm: {np.linalg.norm(natural_centroid):.2f}')
print(f'AI-design centroid norm: {np.linalg.norm(ai_centroid):.2f}')
print(f'Centroid cosine distance: {cosine(natural_centroid, ai_centroid):.4f}')

In [ ]:
# FoldMark Detector: classify a query sequence as natural or AI-designed
def foldmark_detect(sequence, name='query'):
    """Use ESM-2 embeddings to classify a sequence as natural vs AI-designed."""
    emb = get_embedding(sequence, name)
    
    # Cosine similarity to each centroid
    sim_natural = 1 - cosine(emb, natural_centroid)
    sim_ai = 1 - cosine(emb, ai_centroid)
    
    # Also compute similarity to each reference
    natural_sims = [1 - cosine(emb, v) for v in natural_embeddings.values()]
    ai_sims = [1 - cosine(emb, v) for v in ai_embeddings.values()]
    
    classification = 'AI-designed' if sim_ai > sim_natural else 'Natural'
    confidence = abs(sim_ai - sim_natural) / max(sim_ai, sim_natural) * 100
    
    return {
        'classification': classification,
        'confidence': round(confidence, 1),
        'sim_natural_centroid': round(sim_natural, 4),
        'sim_ai_centroid': round(sim_ai, 4),
        'nearest_natural': list(natural_proteins.keys())[np.argmax(natural_sims)],
        'nearest_ai': list(ai_designed.keys())[np.argmax(ai_sims)],
        'max_natural_sim': round(max(natural_sims), 4),
        'max_ai_sim': round(max(ai_sims), 4),
    }

# Test: classify a known natural protein
result_nat = foldmark_detect(
    'MNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKSELDKAIGRNTNGVITKDEAEKLFNQDVDAAVRGILRNAKLKPVYDSLDAVRRAALINMVFQMGETGVAGFTNSLRMLQQKRWDEAAVNLAKSRWYNQTPNRAKRVITTFRTGTWDAYKNL',
    'TrpA_natural'
)
print('=== Natural protein (TrpA synthase) ===')
for k, v in result_nat.items():
    print(f'  {k}: {v}')

print()

# Test: classify an AI-designed sequence
result_ai = foldmark_detect(
    'EELKKEIEELKKEIEELKKEIEELKKEIEELKKEIEELKKEIEELKK',
    'AI_test'
)
print('=== AI-designed sequence ===')
for k, v in result_ai.items():
    print(f'  {k}: {v}')

In [ ]:
---
## Part 3: Threat Screening with ESM-2 Embeddings (GPU)

Instead of simple string matching (which can be evaded), we use **embedding-based screening**. If a designed sequence is semantically similar to a known toxin or virulence factor in ESM-2 embedding space, it gets flagged — even if the exact sequence differs.

This is far more powerful than BLAST-based screening because it captures functional similarity, not just sequence identity.

# Threat reference database — compute embeddings for known dangerous proteins
# These are real N-terminal fragments from public databases (truncated for safety)
threat_db = {
    'Botulinum toxin (fragment)':     'MPFVNKQFNYKDPVNGVDIAYIKIPNAGQMQPVKAFKIHNKIWVIPERDTFTNPEEGDLN',
    'Ricin A chain (fragment)':       'IFPKQYPIINFTTAGATVQSYTNFIRAVRGRLTTGADVRHEIPVLPNRVGLPINQRFIL',
    'Diphtheria toxin (fragment)':    'GADDVVDSSKSFVMENFSSYHGTKPGYVDSIQKGIQKPKSGTQGNYDDDWKGFYSTDN',
    'Staphylococcal enterotoxin':     'ESQPDPKPDELHKSSKFTGLMENMKVLYDDNHVSAINVKSIDQFLYFDLIYSIKDTK',
    'Shiga toxin (fragment)':         'MKCILFKSVLFVSFLANNIYAEFTLDFSTAKTYVDSLNVIRSAIGTPLQTISSGGTS',
}

# Safe reference — common benign/therapeutic proteins
safe_db = {
    'Insulin':          'MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKT',
    'Trastuzumab (Ab)': 'EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTR',
    'Interferon alpha':  'CDLPQTHSLGSRRTLMLLAQMRKISLFSCLKDRHDFGFPQEEFGNQFQKAETIPVLHEM',
    'Erythropoietin':   'APPRLICDSRVLERYLLEAKEAENITTGCAEHCSLNENITVPDTKVNFYAWKRMEVGQQA',
}

print('Computing threat database embeddings (GPU)...')
threat_embeddings = {}
for name, seq in threat_db.items():
    threat_embeddings[name] = get_embedding(seq, name)
    print(f'  ⚠ {name}')

safe_embeddings = {}
for name, seq in safe_db.items():
    safe_embeddings[name] = get_embedding(seq, name)
    print(f'  ✓ {name}')

print(f'\nThreat DB: {len(threat_embeddings)} entries')
print(f'Safe DB: {len(safe_embeddings)} entries')

In [ ]:
def biosecurity_screen(sequence, name='query', threshold=0.85):
    """Screen a protein sequence against threat and safe databases using ESM-2 embeddings.
    
    Returns a full screening report with similarity scores.
    A high similarity to threat proteins (>threshold) triggers a flag.
    """
    query_emb = get_embedding(sequence, name)
    
    # Compare against threat DB
    threat_scores = {}
    for tname, temb in threat_embeddings.items():
        threat_scores[tname] = 1 - cosine(query_emb, temb)
    
    # Compare against safe DB
    safe_scores = {}
    for sname, semb in safe_embeddings.items():
        safe_scores[sname] = 1 - cosine(query_emb, semb)
    
    max_threat = max(threat_scores.values())
    max_safe = max(safe_scores.values())
    nearest_threat = max(threat_scores, key=threat_scores.get)
    nearest_safe = max(safe_scores, key=safe_scores.get)
    
    # Flag if too similar to any threat protein
    flags = []
    for tname, score in threat_scores.items():
        if score > threshold:
            flags.append(f'CRITICAL: High similarity ({score:.3f}) to {tname}')
    
    if max_threat > max_safe:
        flags.append(f'WARNING: More similar to threat DB than safe DB')
    
    passed = len([f for f in flags if f.startswith('CRITICAL')]) == 0
    
    return {
        'passed': passed,
        'flags': flags,
        'max_threat_similarity': round(max_threat, 4),
        'nearest_threat': nearest_threat,
        'max_safe_similarity': round(max_safe, 4),
        'nearest_safe': nearest_safe,
        'all_threat_scores': {k: round(v, 4) for k, v in threat_scores.items()},
        'all_safe_scores': {k: round(v, 4) for k, v in safe_scores.items()},
    }

# Screen our example capstone binder
print('Screening capstone binder sequence...\n')
capstone_seq = 'MKTLLILAVFCLASQCGAELREACKDGELSCADKSKGKCEAKNKKECPSVFNETCEPCR'
report = biosecurity_screen(capstone_seq, 'Lassa_binder_candidate')

print('=' * 60)
print('BIOSECURITY SCREENING REPORT')
print('=' * 60)
print(f'Sequence: {capstone_seq[:40]}...')
print(f'Length: {len(capstone_seq)} aa')
print(f'Result: {"✅ CLEARED" if report["passed"] else "❌ FLAGGED"}')
print(f'\nNearest threat: {report["nearest_threat"]} (sim={report["max_threat_similarity"]})')
print(f'Nearest safe:   {report["nearest_safe"]} (sim={report["max_safe_similarity"]})')

if report['flags']:
    print(f'\nFlags ({len(report["flags"])}):')
    for f in report['flags']:
        print(f'  ⚠ {f}')
else:
    print('\nNo flags raised.')

print('\n--- All threat similarity scores ---')
for name, score in report['all_threat_scores'].items():
    bar = '█' * int(score * 40)
    print(f'  {name:35s} {score:.4f} {bar}')

---
## Part 4: ESMFold Final Validation of Capstone Candidates (GPU)

Re-fold your top capstone candidates with ESMFold to produce final structural QC metrics. This is the last computational checkpoint before recommending sequences for DNA synthesis.

**ESMFold** is a single-sequence structure predictor (~700M parameters). It runs entirely on GPU and produces pLDDT confidence scores per residue.

In [ ]:
# Load ESMFold for structural validation (GPU-intensive: ~3GB VRAM)
# Free the ESM-2 model first to make room
del model
torch.cuda.empty_cache()

print('Loading ESMFold... (this takes ~60 seconds)')
esmfold = esm.pretrained.esmfold_v1()
esmfold = esmfold.eval().cuda()
print('ESMFold loaded and ready.')

from Bio.PDB import PDBParser
parser = PDBParser(QUIET=True)

def validate_with_esmfold(sequence, name='candidate'):
    """Fold a sequence with ESMFold and compute per-residue pLDDT."""
    with torch.no_grad():
        pdb_string = esmfold.infer_pdb(sequence)
    
    # Save PDB
    pdb_path = f'/content/{name}.pdb'
    with open(pdb_path, 'w') as f:
        f.write(pdb_string)
    
    # Extract pLDDT from B-factors
    structure = parser.get_structure(name, pdb_path)
    plddt_scores = [a.get_bfactor() for a in structure.get_atoms() if a.get_name() == 'CA']
    
    return {
        'mean_plddt': round(np.mean(plddt_scores), 1),
        'min_plddt': round(np.min(plddt_scores), 1),
        'max_plddt': round(np.max(plddt_scores), 1),
        'per_residue': plddt_scores,
        'pdb_path': pdb_path,
        'length': len(sequence),
    }

# =============================================================
# REPLACE THESE WITH YOUR TOP 3 CAPSTONE SEQUENCES FROM MODULE 4
# =============================================================
capstone_candidates = {
    'LassaBinder_1': 'EEEELKKELKKELKKELKKELKKELKKELKKELKKELKKELKKELKKELKKELKKEEEE',
    'LassaBinder_2': 'DAAARKAAARKAAARKAAARKAAARKAAARKAAARKAAARKAAARKAAARKAAARKDDD',
    'LassaBinder_3': 'SIEELKKEIEELKKEIEELKKEIEELKKEIEELKKEIEELKKEIEELKKEIEELKKS',
}

print(f'\nValidating {len(capstone_candidates)} capstone candidates with ESMFold...\n')

validation_results = {}
for name, seq in capstone_candidates.items():
    print(f'Folding {name} ({len(seq)} aa)...', end=' ')
    result = validate_with_esmfold(seq, name)
    validation_results[name] = result
    status = '✓ PASS' if result['mean_plddt'] > 80 else '✗ FAIL'
    print(f'pLDDT = {result["mean_plddt"]} {status}')

print('\n=== FINAL VALIDATION SUMMARY ===')
for name, r in validation_results.items():
    print(f'{name}: pLDDT={r["mean_plddt"]} (min={r["min_plddt"]}, max={r["max_plddt"]})')

# Per-residue pLDDT visualization for all candidates
fig, axes = plt.subplots(len(validation_results), 1, figsize=(12, 3*len(validation_results)), sharex=False)
if len(validation_results) == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, validation_results.items()):
    residues = range(1, len(r['per_residue']) + 1)
    colors = ['#22c55e' if p > 90 else '#eab308' if p > 70 else '#ef4444' for p in r['per_residue']]
    ax.bar(residues, r['per_residue'], color=colors, width=1.0, edgecolor='none')
    ax.axhline(y=80, color='blue', linestyle='--', alpha=0.3, label='pLDDT=80 threshold')
    ax.axhline(y=90, color='green', linestyle='--', alpha=0.3, label='pLDDT=90 (high confidence)')
    ax.set_ylabel('pLDDT')
    ax.set_title(f'{name} — Mean pLDDT: {r["mean_plddt"]}', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Residue Position')
plt.suptitle('ESMFold Validation — Per-Residue Confidence', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/esmfold_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Validation plot saved to /content/esmfold_validation.png')

In [ ]:
---
## Part 5: Embedding Space Visualization (GPU)

Visualize where your capstone candidates sit in ESM-2 embedding space relative to natural proteins, AI designs, and known threats. This gives an intuitive sense of how your designs "look" to a protein language model.

We use **PCA** to reduce the 1280-dimensional ESM-2 embeddings to 2D for plotting.

# PCA visualization of embedding space
from sklearn.decomposition import PCA

# Compute embeddings for capstone candidates (re-load ESM-2 lite or use cached)
# We'll use the embeddings already computed by the screening function
capstone_embeddings = {}
for name, seq in capstone_candidates.items():
    capstone_embeddings[name] = get_embedding(seq, name)

# Combine all embeddings for PCA
all_labels, all_embeddings, all_categories = [], [], []

for name, emb in natural_embeddings.items():
    all_labels.append(name)
    all_embeddings.append(emb)
    all_categories.append('Natural')

for name, emb in ai_embeddings.items():
    all_labels.append(name)
    all_embeddings.append(emb)
    all_categories.append('AI-Designed')

for name, emb in threat_embeddings.items():
    all_labels.append(name)
    all_embeddings.append(emb)
    all_categories.append('Threat')

for name, emb in capstone_embeddings.items():
    all_labels.append(name)
    all_embeddings.append(emb)
    all_categories.append('Your Capstone')

# PCA to 2D
X = np.array(all_embeddings)
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
color_map = {'Natural': '#22c55e', 'AI-Designed': '#3b82f6', 'Threat': '#ef4444', 'Your Capstone': '#f59e0b'}
marker_map = {'Natural': 'o', 'AI-Designed': 's', 'Threat': 'X', 'Your Capstone': '*'}
size_map = {'Natural': 80, 'AI-Designed': 80, 'Threat': 120, 'Your Capstone': 200}

for cat in ['Natural', 'AI-Designed', 'Threat', 'Your Capstone']:
    mask = [c == cat for c in all_categories]
    xs = X_2d[mask, 0]
    ys = X_2d[mask, 1]
    ax.scatter(xs, ys, c=color_map[cat], marker=marker_map[cat], 
               s=size_map[cat], label=cat, alpha=0.8, edgecolors='white', linewidths=0.5)
    
    # Label capstone candidates
    if cat == 'Your Capstone':
        labels_subset = [l for l, m in zip(all_labels, mask) if m]
        for x, y, label in zip(xs, ys, labels_subset):
            ax.annotate(label, (x, y), fontsize=9, fontweight='bold',
                       xytext=(5, 5), textcoords='offset points')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
ax.set_title('ESM-2 Embedding Space — Where Do Your Designs Sit?', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/content/embedding_space.png', dpi=150)
plt.show()

print(f'PCA explained variance: {pca.explained_variance_ratio_[0]*100:.1f}% + {pca.explained_variance_ratio_[1]*100:.1f}%')
print('Your capstone candidates should cluster AWAY from the Threat zone.')

In [ ]:
# Export final biosecurity report + validated sequences
import json

final_report = {
    'bootcamp': 'Lagos Bio-Design Bootcamp — Cohort 1',
    'module': 'Module 5: Biosecurity, Ethics & The Future',
    'timestamp': datetime.datetime.now().isoformat(),
    'candidates': {}
}

for name, seq in capstone_candidates.items():
    # Run screening
    screen = biosecurity_screen(seq, name)
    # Run FoldMark detection
    detect = foldmark_detect(seq, name)
    # Get validation
    val = validation_results.get(name, {})
    
    final_report['candidates'][name] = {
        'sequence': seq,
        'length': len(seq),
        'esmfold_plddt': val.get('mean_plddt', 'N/A'),
        'biosecurity_passed': screen['passed'],
        'max_threat_similarity': screen['max_threat_similarity'],
        'nearest_threat': screen['nearest_threat'],
        'foldmark_classification': detect['classification'],
        'foldmark_confidence': detect['confidence'],
        'audit_hash': hashlib.sha256(seq.encode()).hexdigest()[:16],
    }

# Save report
with open('/content/biosecurity_report.json', 'w') as f:
    json.dump(final_report, f, indent=2)

# Save sequences as FASTA
with open('/content/final_candidates.fasta', 'w') as f:
    for name, data in final_report['candidates'].items():
        f.write(f'>{name}_pLDDT{data["esmfold_plddt"]}_hash{data["audit_hash"]}\n')
        f.write(f'{data["sequence"]}\n')

# Print summary
print('=' * 60)
print('FINAL BIOSECURITY + VALIDATION REPORT')
print('=' * 60)
for name, data in final_report['candidates'].items():
    screen_status = '\u2705 CLEARED' if data['biosecurity_passed'] else '\u274c FLAGGED'
    plddt_status = '\u2705' if isinstance(data['esmfold_plddt'], (int, float)) and data['esmfold_plddt'] > 80 else '\u26a0\ufe0f'
    print(f'\n--- {name} ---')
    print(f'  ESMFold pLDDT:  {data["esmfold_plddt"]} {plddt_status}')
    print(f'  Biosecurity:    {screen_status}')
    print(f'  Threat sim:     {data["max_threat_similarity"]} (nearest: {data["nearest_threat"]})')
    print(f'  FoldMark:       {data["foldmark_classification"]} ({data["foldmark_confidence"]}% confidence)')
    print(f'  Audit hash:     {data["audit_hash"]}')

# Download
from google.colab import files
files.download('/content/biosecurity_report.json')
files.download('/content/final_candidates.fasta')
files.download('/content/embedding_space.png')
files.download('/content/esmfold_validation.png')
print('\nAll files downloaded.')

In [ ]:
---
## Summary

In this final module you used **GPU-powered protein language models** to audit your capstone designs:

1. **ESM-2 (650M params)** — Computed sequence embeddings for AI-design detection and threat screening
2. **FoldMark Detection** — Classified sequences as natural vs AI-designed using embedding centroids
3. **Biosecurity Screening** — Screened against known toxin/pathogen embeddings (not just string matching)
4. **ESMFold Validation** — Re-folded top candidates for final structural QC with per-residue pLDDT
5. **Embedding Space Visualization** — PCA plot showing where your designs sit relative to threats

### GPU Resources Used
| Model | Parameters | VRAM | Purpose |
|-------|-----------|------|---------|
| ESM-2 | 650M | ~2.5 GB | Sequence embeddings, threat screening |
| ESMFold | ~700M | ~3 GB | Structural validation, pLDDT |

### Key Takeaways
- **Embedding-based screening** catches functional similarity that BLAST misses
- **AI-designed proteins have detectable signatures** in ESM-2 embedding space
- **Every sequence destined for synthesis must pass biosecurity screening**
- **Audit trails** (hashes, timestamps, designer info) are non-negotiable

### Deliverable
Submit your **biosecurity_report.json** along with a **5-minute pitch deck** covering:
1. The problem (Lassa fever)
2. Your solution (AI-designed binder)
3. Your science (pLDDT, RMSD, biosecurity clearance)
4. Market & team
5. The ask (seed funding for lab validation)

---

**Congratulations on completing the Lagos Bio-Design Bootcamp!** 🎉

You can now design proteins computationally, validate them in-silico, screen them for biosecurity, and present them as viable biotech innovations. The future of African biotech starts with you.

---
## Part 7: Ethics Self-Assessment Checklist

Complete this checklist before submitting your capstone. Every responsible bio-designer should be able to answer these questions.

In [ ]:
ethics_checklist = [
    ('Dual-Use Assessment', 'Could your designed protein be misused for harmful purposes?'),
    ('Biosecurity Screening', 'Have you screened your sequences against known threat databases?'),
    ('Informed Purpose', 'Is the intended application clearly documented and beneficial?'),
    ('Audit Trail', 'Can your design be traced back to you, your tools, and your rationale?'),
    ('Data Provenance', 'Are your training structures from legitimate, public sources (PDB)?'),
    ('Equity Consideration', 'Will the end product be accessible to those who need it most?'),
    ('Environmental Impact', 'Have you considered the ecological implications of your design?'),
    ('IP Transparency', 'Have you documented which AI tools contributed to the design?'),
    ('Regulatory Awareness', 'Do you know which regulatory bodies govern your application?'),
    ('Community Benefit', 'Does your design address a genuine need in the local context?'),
]

print('=' * 60)
print('ETHICS SELF-ASSESSMENT CHECKLIST')
print('Lagos Bio-Design Bootcamp — Module 5')
print('=' * 60)
print()

# ====================================
# EDIT: Set True/False for each item
# ====================================
responses = [True, True, True, True, True, True, False, True, True, True]

passed = 0
for i, ((title, question), response) in enumerate(zip(ethics_checklist, responses), 1):
    status = '\u2705' if response else '\u274c'
    print(f'{status} {i:2d}. {title}')
    print(f'     {question}')
    if response:
        passed += 1

print(f'\nScore: {passed}/{len(ethics_checklist)}')
if passed == len(ethics_checklist):
    print('\u2728 All checks passed. Your design meets ethical standards.')
elif passed >= 8:
    print('\u26a0\ufe0f Almost there. Address the unchecked items before submission.')
else:
    print('\u274c Several items need attention. Review your design process.')

---
## Summary

In this final module you explored the responsibilities that come with generative protein design:

1. **FoldMark Audit Trail** — Created traceable records for AI-designed proteins
2. **Biosecurity Screening** — Implemented sequence screening against threat databases
3. **Open vs. Closed Debate** — Analysed trade-offs from the African lab perspective
4. **Intellectual Property** — Documented creative decisions to support IP claims
5. **Regulatory Landscape** — Mapped the path from in-silico design to NAFDAC approval
6. **Pitch Compilation** — Compiled capstone results into a startup pitch
7. **Ethics Checklist** — Self-assessed responsible design practices

### Key Takeaways
- **Power comes with responsibility.** The same tools that design diagnostics could theoretically design threats.
- **Screening is non-negotiable.** Every sequence destined for synthesis must be screened.
- **Africa needs its own biosecurity infrastructure.** Don't wait for Western frameworks — build local ones.
- **Document everything.** Audit trails protect you and the community.

### Deliverable
Submit a **5-minute pitch video** or **PDF slide deck** covering:
1. The problem (Lassa fever)
2. Your solution (AI-designed binder)
3. Your science (pLDDT, RMSD, screening results)
4. Market & team
5. The ask (seed funding for lab validation)

---

**Congratulations on completing the Lagos Bio-Design Bootcamp!** 🎉

You now have the skills to design proteins computationally, validate them in-silico, and present them as viable biotech innovations. The future of African biotech starts with you.